In [12]:
import polars as pl
from pathlib import Path

trec_root = Path("~/scratch/trec-tot-2025").expanduser()
dataset_root = trec_root / "data/enwiki/processed"
gdrive_path = trec_root / "data/gdrive/data"
official_path = trec_root / "data/official"
output_path = trec_root / "results/rerank"


run_path = f"{gdrive_path}/shared_retrieval_results/gemini-2.5-flash/dev3.run"
rerank_part_path = f"{dataset_root}/reranked/v2/bge-m3-knn-k15/ppr"

rerankdf = pl.read_parquet(f"{rerank_part_path}/*.parquet")
rerankdf.collect_schema()
rerankdf.head()

qid,Q0,docid,rank,score,run_name
i64,str,i64,i64,f64,str
2008,"""Q0""",7079011,2,0.047599,"""gemini-25_alias"""
2008,"""Q0""",52631552,0,0.031995,"""gemini-25_alias"""
2008,"""Q0""",34726199,5,0.031069,"""gemini-25_alias"""
2008,"""Q0""",30517310,3,0.030168,"""gemini-25_alias"""
2008,"""Q0""",26600762,4,0.030139,"""gemini-25_alias"""


In [10]:
# write out the reranked result to a trec styled file
reranked_path = f"{output_path}/gemini-2.5-flash-dev3-ppr-knn-k15-v1/results.txt"
Path(reranked_path).parent.mkdir(parents=True, exist_ok=True)
# space as separator, no index, no header
rerankdf.write_csv(reranked_path, include_header=False, separator=" ")
! head {reranked_path}

2008 Q0 7079011 2 0.047599364669133545 gemini-25_alias
2008 Q0 52631552 0 0.03199522943524085 gemini-25_alias
2008 Q0 34726199 5 0.03106864217537808 gemini-25_alias
2008 Q0 30517310 3 0.030167610180709785 gemini-25_alias
2008 Q0 26600762 4 0.03013861411186509 gemini-25_alias
2008 Q0 462318 1 0.027737450494380887 gemini-25_alias
2009 Q0 19802830 21 0.011202623466013263 gemini-25_alias
2009 Q0 25556232 17 0.008592720151108405 gemini-25_alias
2009 Q0 2455341 19 0.008147847631191222 gemini-25_alias
2009 Q0 2675451 11 0.008004101994585955 gemini-25_alias


In [15]:
! trec_eval \
    -m ndcg_cut.10,1000 \
    -m recall.1000 \
    -m recip_rank \
    -c \
    {official_path}/dev3-2025-qrel.txt \
    {gdrive_path}/shared_retrieval_results/gemini-2.5-flash/dev3.run

recip_rank            	all	0.3001
recall_1000           	all	0.4403
ndcg_cut_10           	all	0.3267
ndcg_cut_1000         	all	0.3327


In [16]:
! trec_eval \
    -m ndcg_cut.10,1000 \
    -m recall.1000 \
    -m recip_rank \
    -c \
    {official_path}/dev3-2025-qrel.txt \
    {reranked_path} 

recip_rank            	all	0.1089
recall_1000           	all	0.4328
ndcg_cut_10           	all	0.1459
ndcg_cut_1000         	all	0.1784
